In [1]:
import sys
# sys.path.insert(0, '../../Utilities/')

# from jax.example_libraries import optimizers
import jax.numpy as jnp
import numpy as np
from jax import grad, jit, vmap, value_and_grad, hessian
from jax import random
from functools import partial
import flax.linen as nn
import jax
import optax
import pickle
from datetime import datetime
import os
import numpy as np
from scipy import sparse
import time
from jax import grad, hessian, jit, vmap, pmap
from jax.tree_util import tree_leaves
from datetime import datetime
import matplotlib.pyplot as plt
import pickle
import flax.linen as nn
import optax
import optax.tree_utils as otu

from mg_pinn_class import MinMaxOptProb

In [2]:
# Define boundary and initial conditions
def u0(x):
    # return 2*smooth_sign(x) - 1
    return -jnp.sin(jnp.pi * x)
    # return -jnp.sin(jnp.pi*x)
# def u_exact(tx):
#     t, x = tx[0], tx[1]
#     return u0(x-0.5*t)
# vec_u_exact = vmap(u_exact)

# Boundary at x=-1
def u1(x):
    return 0

# Boundary at x=1
def u2(x):
    return 0


def res_visc(u_quad, u_t, u_x, u_xx, nu, u_exact=0):
    # print(f"u_quad shape: {u_quad.shape}")
    # print(f"u2_x shape: {u2_x.shape}")
    # print(f"u_xx shape: {u_xx.shape}")
    return u_t + u_quad*u_x - nu*u_xx
    # return u_t + 0.8*u_x - nu*u_xx
    # return jnp.zeros_like(u_quad)
    # return u_quad - u_exact

def res(u_quad, u_t, u_x):
    return u_t + u_quad * u_x
    # return u_t + 0.8*u_x

# Initialize separate loss history lists for each stage
pre_loss_int_history = []
pre_loss_bc_history = []
train_loss_int_history = []
train_loss_bc_history = []

# Initialize the optimization problem
optProb = MinMaxOptProb(
    N_t=16,
    N_x=32,
    lr_theta=1e-3,
    N_bc=100,
    N_ic=200,
    t_min=0, t_max=1, x_min=-1, x_max=1,
    hidden_dims_list=[(40, 40), (40, 40), (40, 40)],
    seed=0,
    u0=u0, u1=u1, u2=u2,
    nu=0.01,
    res_visc=res_visc,
    res=res,    
)

In [3]:
optProb.pre_train_process()


Starting Pre-Train Stage with MGDL


wandb: Currently logged in as: haoli1 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Training Grade 0
Grade 0, Iteration 0, lambda_ic: 1.94635367, lambda_bc: 2.12740064, lambda_r: 1.02151299
Grade 0, Epoch 0, loss: 1.00612450, loss_eqn: 0.44993654, loss_ibc: 0.55618793
Grade 0, Epoch 100, loss: 0.42658079, loss_eqn: 0.01086177, loss_ibc: 0.41571900
Grade 0, Epoch 200, loss: 0.21973246, loss_eqn: 0.11970100, loss_ibc: 0.10003146
Grade 0, Epoch 300, loss: 0.19054995, loss_eqn: 0.10668404, loss_ibc: 0.08386591
Grade 0, Epoch 400, loss: 0.15636101, loss_eqn: 0.09164346, loss_ibc: 0.06471755
Grade 0, Epoch 500, loss: 0.12983841, loss_eqn: 0.07544755, loss_ibc: 0.05439085
Grade 0, Epoch 600, loss: 0.11441762, loss_eqn: 0.06777048, loss_ibc: 0.04664714
Grade 0, Epoch 700, loss: 0.10314269, loss_eqn: 0.06366461, loss_ibc: 0.03947809
Grade 0, Epoch 800, loss: 0.08672193, loss_eqn: 0.05284566, loss_ibc: 0.03387627
Grade 0, Epoch 900, loss: 0.06576152, loss_eqn: 0.03971635, loss_ibc: 0.02604517
Grade 0, Iteration 1000, lambda_ic: 2.03227782, lambda_bc: 3.36150289, lambda_r: 1.09

grade_0_loss_eqn,█▁▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
grade_0_loss_ibc,█▆▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
grade_1_loss_eqn,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
grade_1_loss_ibc,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
grade_2_loss_eqn,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
grade_2_loss_ibc,▇▇▇▆▆▆▆▆▆▆▆▆▆█▇▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁
second_stage_loss,███████▇███▇▇▇▇▇▆▆▅▅▅▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁
step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▆▆▆▆▆▆▆▇▇▇███
grade_0_loss_eqn,0.00851
grade_0_loss_ibc,0.00457
grade_1_loss_eqn,0.01886


In [4]:
optProb.model